# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² mental health survey dataset from Kilifi County, Kenya, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL (FAIR² Croissant package):
```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant` and review top-level information.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Dataset Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the Croissant metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n{metadata.description}")

## 2. Data Overview
Review available record sets defined in the dataset, along with their `@id`s, associated fields, and their `@id`s.

Entities are referenced via their Croissant `@id` for clarity and reproducibility.

In [ ]:
print("Available Record Sets with their @id and fields:")

record_sets = list(metadata.record_sets)
for rec_set in record_sets:
    print(f"- Record Set: {rec_set.name} | @id: {rec_set.id}")
    print("  Fields:")
    for field in rec_set.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print("")

## 3. Data Extraction
Load data from a selected record set into a pandas DataFrame for analysis.

Use the record set and field `@id`s identified in the previous cell. The main survey data is typically under the primary record set, such as demographic or psychological indicators.

In [ ]:
# List all record set @id values
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

# Load records for each record set into a DataFrame
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Show info for the primary record set (choose the main survey record set, e.g. the one with most fields):
main_record_set_id = max(record_set_ids, key=lambda rs_id: len(dataframes[rs_id].columns))
print(f"Primary record set @id: {main_record_set_id}")
print("Columns (fields @id):", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records by a numerical field (e.g., GAD-7, PHQ-9 total scores), normalizing values, and grouping by categorical variables (e.g., gender or education level).

All field references use the field `@id` from the overview above.

In [ ]:
# For this example, select the GAD-7 total score numeric field by its @id.
# Replace with the correct @id as listed in your record set fields (example below is an educated guess).

# Identify candidate numeric field IDs (e.g. GAD-7 total @id, PHQ-9 total @id)
candidate_numeric_fields = [col for col in dataframes[main_record_set_id].columns if 'gad' in col.lower() or 'phq' in col.lower() or 'psq' in col.lower() or 'score' in col.lower()]
print(f"Numeric candidate fields: {candidate_numeric_fields}")

# Set a default if found, or set manually if known
numeric_field = candidate_numeric_fields[0] if candidate_numeric_fields else dataframes[main_record_set_id].columns[0]

threshold = 10

df = dataframes[main_record_set_id]
if pd.api.types.is_numeric_dtype(df[numeric_field]):
    filtered_df = df[df[numeric_field] > threshold].copy()
else:
    # Attempt conversion if not already numeric
    filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold].copy()
    filtered_df[numeric_field] = pd.to_numeric(filtered_df[numeric_field], errors='coerce')

print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize
if len(filtered_df) > 0:
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field such as 'gender' or 'level_of_education' by their @id
    possible_category_fields = [c for c in df.columns if any(substring in c.lower() for substring in ['gender', 'education', 'sex', 'village', 'marital'])]
    group_field = possible_category_fields[0] if possible_category_fields else None

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
else:
    print("No records above threshold were found for analysis.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship to a categorical field, using the field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=20)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.show()

# If there's a categorical field, plot a boxplot
if 'group_field' in locals() and group_field:
    plt.figure(figsize=(12,5))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² mental health survey dataset from Kilifi County, Kenya. Using the `mlcroissant` library, we loaded Croissant metadata, programmatically inspected available record sets and fields (referenced by their `@id`), and performed preliminary data extraction and statistical analysis.

**Key steps:**
- Programmatic record set and field enumeration via `@id`
- Data extraction into DataFrames
- Numeric data filtering, normalization, and aggregation by categorical variables
- Visualization of selected indicators

For deeper analysis, consult the full field documentation (IDs and descriptions) in the Croissant schema, and iterate on grouping, filtering, or feature engineering tasks as appropriate to your research.